# 03 — LUNA16 classical ML (SVM / RF / KNN)

5-fold stratified outer CV with inner 3-fold grid search per model. Both default-threshold and Youden-calibrated metrics are saved per fold to `results/luna16_<model>.json`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import yaml

from utils.seed import set_seed
from utils.models import CLASSICAL_REGISTRY
from utils.training import train_classical_cv, save_fold_results
from utils.metrics import aggregate_folds
from utils.stats import format_results_table

with open('../configs/luna16.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
data = np.load(cache_dir / 'features.npz', allow_pickle=True)
X, y = data['X'], data['y']
groups = data['seriesuid']  # GroupKFold by CT scan to avoid patch-level leakage
print('X', X.shape, 'pos rate:', y.mean())

X (448, 91) pos rate: 0.25


In [2]:
results_dir = Path('..') / cfg['paths']['results']
results_dir.mkdir(parents=True, exist_ok=True)

aggregated = {}
for name, (builder, grid) in CLASSICAL_REGISTRY.items():
    print(f'\n=== {name} ===')
    folds = train_classical_cv(
        X, y, builder, grid,
        n_splits=cfg['cv']['n_splits'],
        inner_splits=cfg['cv']['inner_splits'],
        scoring=cfg['cv']['scoring'],
        seed=cfg['seed'],
        groups=groups,
        verbose=True,
    )
    save_fold_results(folds, results_dir / f'luna16_{name}.json')
    aggregated[name] = aggregate_folds([f.metrics_calibrated for f in folds])


=== SVM ===
[fold 0] train=356 test=92


[fold 0] best_params={'C': 10, 'gamma': 'scale', 'kernel': 'linear'} thr=0.116 AUC=0.998 BalAcc(cal)=0.971
[fold 1] train=356 test=92
[fold 1] best_params={'C': 1, 'gamma': 'scale', 'kernel': 'linear'} thr=0.033 AUC=1.000 BalAcc(cal)=0.978
[fold 2] train=360 test=88
[fold 2] best_params={'C': 1, 'gamma': 0.01, 'kernel': 'rbf'} thr=0.406 AUC=1.000 BalAcc(cal)=1.000
[fold 3] train=360 test=88
[fold 3] best_params={'C': 1, 'gamma': 'scale', 'kernel': 'linear'} thr=0.026 AUC=0.992 BalAcc(cal)=0.977
[fold 4] train=360 test=88
[fold 4] best_params={'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'} thr=0.781 AUC=0.999 BalAcc(cal)=0.977

=== RF ===
[fold 0] train=356 test=92
[fold 0] best_params={'max_depth': None, 'min_samples_split': 2, 'n_estimators': 500} thr=0.600 AUC=1.000 BalAcc(cal)=1.000
[fold 1] train=356 test=92
[fold 1] best_params={'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200} thr=0.650 AUC=1.000 BalAcc(cal)=1.000
[fold 2] train=360 test=88
[fold 2] best_params={'m

In [3]:
table = format_results_table(aggregated)
table

,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,auc_roc,pr_auc
model,,,,,,,,
SVM,0.985 ± 0.012,0.973 ± 0.025,0.988 ± 0.019,0.968 ± 0.050,0.970 ± 0.023,0.981 ± 0.011,0.998 ± 0.003,0.995 ± 0.007
RF,0.995 ± 0.006,0.982 ± 0.025,1.000 ± 0.000,1.000 ± 0.000,0.991 ± 0.013,0.991 ± 0.012,0.994 ± 0.011,0.991 ± 0.015
KNN,0.973 ± 0.019,0.892 ± 0.076,1.000 ± 0.000,1.000 ± 0.000,0.942 ± 0.044,0.946 ± 0.038,0.980 ± 0.011,0.972 ± 0.016


In [4]:
import json

for name, (_, grid) in CLASSICAL_REGISTRY.items():
    folds_path = results_dir / f'luna16_{name}.json'
    with folds_path.open() as f:
        raw = json.load(f)
    print(name, 'selected hparams per fold:')
    for r in raw:
        print(' ', r['fold'], r['best_params'])

SVM selected hparams per fold:
  0 {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
  1 {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
  2 {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
  3 {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
  4 {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
RF selected hparams per fold:
  0 {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 500}
  1 {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
  2 {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
  3 {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
  4 {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
KNN selected hparams per fold:
  0 {'n_neighbors': 7, 'p': 1, 'weights': 'distance'}
  1 {'n_neighbors': 7, 'p': 1, 'weights': 'uniform'}
  2 {'n_neighbors': 21, 'p': 2, 'weights': 'distance'}
  3 {'n_neighbors': 7, 'p': 1, 'weights': 'distance'}
  4 {'n_neighbors': 3, 'p': 1, 'weights': 'distance'}
